[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Why Peewee


## What you will be able to do

Say what an object-relational mapper buys you once the query it replaces has been written correctly,
which is the only comparison worth making. Read a query peewee builds as the SQL it will send, get a
row back as an object with attributes rather than a tuple to index, declare a table once in Python,
and reach a related row by attribute. Say which three things SQLAlchemy does that peewee does not,
and recognize the mistakes that come from the two habits a reader arrives with, Django's and the
standard library's.


## The idea

### The problem

A script that reads a database starts simply. It opens a connection, sends a `SELECT` with the name
of an author in it, and prints what comes back. The SQL is a string, and building a string with a
value in it is what everybody does first:

    "SELECT * FROM author WHERE name = '%s'" % name

That works until a name has an apostrophe in it. Then the quote the name carries ends the quote the
query opened, and the database is handed something that is no longer a query. The failure is loud,
and it is the polite version of the problem: the same line with a name chosen by somebody else is
how a database gets emptied.

The fix is one character and belongs to the standard library, not to any mapper: a `?` where the
value goes, and the value passed beside the query. That is the whole of it, and this notebook shows
it immediately, because an argument for a library has to be made against code that already works.

What is left after the fix is the real question. The row that comes back is a tuple, so every piece
of it is `row[2]` somewhere, and a column added at the front moves everything. The table's shape is
written in one place and the code that reads it in another, so the two drift. And a book's author is
an id, so reaching it is another query written by hand. None of that is a bug. It is what a program
that grows past a script has to keep doing by hand, and it is what peewee does instead.

### What peewee is

> **peewee** is an object-relational mapper: one module, `peewee.py`, with no required dependencies.
> A **model** is a class whose attributes are **fields**, and the class is the table: peewee writes
> the `CREATE TABLE` from it. A **query** is built from the model, such as
> `Book.select().where(Book.year > 2015)`, and until it is run it is an object that can be printed
> as the SQL it will send. What comes back
> is a **model instance**, with the columns as attributes and the related row reachable through the
> foreign key. `playhouse` is the package of extras that ships beside it: full-text search, the
> migration runner, connection pools, and the query counters this guide measures with.

### Why it works that way

- **The query is built, not written.** A value goes in as a value, so a name with an apostrophe is
  just a name, and the SQL is the same shape whatever the value is.
- **The class is the schema.** One declaration makes the table, drives the queries, and is what the
  migration tool compares the database against.
- **A row is an object.** `book.title` rather than `row[1]`, which survives a column being added and
  says what it means at the point it is read.
- **Nothing is hidden that you cannot print.** Every query has a `.sql()`, so a claim about what a
  page costs can be checked rather than believed, which is how this guide makes its claims.
- **It is small on purpose.** One module and no dependencies means it can be read, and it means the
  things it does not do are absent rather than configurable.

### The SQL this guide assumes

This guide is about peewee, not about SQL. It assumes you have met these, and each row says where
peewee's version of it is taught:

| The SQL | What peewee calls it | Where it is taught |
|---|---|---|
| `CREATE TABLE` | a model class, and `create_tables` | the **Models and Fields** notebook |
| `INSERT` | `create`, `save` and `insert_many` | the **Creating and Changing Rows** notebook |
| `SELECT ... WHERE` | `select()` and `where()` | the **Selecting Rows** notebook |
| `ORDER BY`, `LIMIT` | `order_by`, `limit` and `paginate` | the **Selecting Rows** notebook |
| `GROUP BY` with an aggregate | `group_by` with `fn.COUNT` | the **Selecting Rows** notebook |
| `COMMIT` and `ROLLBACK` | `db.atomic()` | the **Transactions** notebook |
| `JOIN` | `join()`, and a foreign key read as an attribute | the **Relationships** notebook |
| a foreign key constraint | `ForeignKeyField`, and one pragma on SQLite | the **Relationships** notebook |
| `ALTER TABLE` | a migration, written or generated | the **Migrations** notebook |
| `CREATE VIRTUAL TABLE ... fts5` | `FTS5Model` and `SearchField` | the **FTS5Model and SearchField** notebook |

The **sqlite3, Deep Dive** guide is where the SQL itself is taught, along with the engine underneath
it: placeholders, the locking model, and full-text search. This guide never explains what a `JOIN`
is; it shows what peewee writes when you ask for one.

### Where this shows up

Scripts that outgrew a dictionary, small web applications, data-loading jobs, anything with a
database and no appetite for configuring one. The **Object-Oriented Python** guide is the
prerequisite: a model is a class, its fields are class attributes, and `Meta` is a class inside a
class. Asynchronous work is not peewee's ground, and the **asyncpg and psycopg3, Deep Dive** guide
is where that belongs.

### What this notebook covers

- A query built by hand, and a name with an apostrophe
- The fix, which is one character and not an argument for anything
- What is left: a row that is an object
- A schema written once
- A related row reached by attribute
- Where SQLAlchemy is the better answer
- A catalog in fifteen lines, finished
- Three failures, two of them habits from somewhere else

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, ForeignKeyField, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Author(Model):
    name = CharField()

    class Meta:
        database = db


class Book(Model):
    title = CharField()
    author = ForeignKeyField(Author, backref="books")

    class Meta:
        database = db


db.create_tables([Author, Book])
ines = Author.create(name="Ines O'Brien")
Book.create(title="A Careful Fire", author=ines)

book = Book.get(Book.title == "A Careful Fire")
print("a row     :", book.title, "|", type(book).__name__)
print("its author:", book.author.name)
print("the SQL   :", Author.select().sql()[0])
```

```
a row     : A Careful Fire | Book
its author: Ines O'Brien
the SQL   : SELECT "t1"."id", "t1"."name" FROM "author" AS "t1"
```

Two classes, two tables, and a book that knows its author. The apostrophe in the name went in and
came out without anybody thinking about it, because the value was never part of the query's text.
And the query is readable before it runs: `.sql()` is what makes every claim in this guide checkable.


## Setup

Seven imports, peewee installed and pinned, the catalog, and two helpers.

- `peewee` is the library, and `Model`, `CharField`, `IntegerField`, `ForeignKeyField` and
  `SqliteDatabase`, from it, are the class, its fields and the database it is bound to
- `sqlite3` is the standard library's driver, which this notebook uses to build the query that
  breaks, and which peewee uses underneath on SQLite. The cell prints the SQLite version beside
  peewee's
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee where the version is not
  4.5.1, which on Colab it is not: Colab ships 4.4.0, and the messages this guide prints in
  **Common errors** are 4.5's, from the release of 8 September 2026
- `re` takes a memory address out of a message
- `AUTHORS` and `BOOKS` are the catalog this guide works on: four authors and twelve books, three
  each, from lists rather than from `random`, so that every count and every ordering is the same on
  every machine. One author is Ines O'Brien, whose apostrophe is what the first worked example
  breaks on
- `sql` prints the SQL a query will send with its values, and `message` prints an error's class and
  text

Every database in this guide is SQLite, and nothing but peewee is installed. The
**SQLite and PostgreSQL** notebook is where the same models meet another backend, without a server.


In [1]:
import re
import sqlite3
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import CharField, DateField, ForeignKeyField, IntegerField, Model, SqliteDatabase

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

def message(error):
    """An error's class and text, without the memory address that makes no two runs agree."""
    return f"{type(error).__module__}.{type(error).__name__}: {re.sub(r'0x[0-9a-f]+', '0x...', str(error))}"

print("peewee", peewee.__version__, "| sqlite", sqlite3.sqlite_version)
print(len(AUTHORS), "authors and", len(BOOKS), "books in the catalog")


peewee 4.5.1 | sqlite 3.50.4
4 authors and 12 books in the catalog


## Worked examples

### A query built by hand, and a name with an apostrophe

Two authors in a table, through the standard library's driver, and a query built the way everybody
builds one first:


In [2]:
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE author (id INTEGER PRIMARY KEY, name TEXT)")
connection.executemany("INSERT INTO author (name) VALUES (?)",
                       [("Ursula Vance",), ("Ines O'Brien",)])

print("looking for Ursula Vance:", connection.execute(
    "SELECT id, name FROM author WHERE name = '%s'" % "Ursula Vance").fetchall())

connection.execute("SELECT id, name FROM author WHERE name = '%s'" % "Ines O'Brien")


looking for Ursula Vance: [(1, 'Ursula Vance')]


OperationalError: near "Brien": syntax error

The first name worked, which is what makes this worth showing: the line is correct for every name
without a quote in it, and most names do not have one. The second ended the string early, so SQLite
was handed `... WHERE name = 'Ines O` followed by `Brien'`, and `near "Brien"` is where it gave up.

A name is the polite version. The same line with a value somebody else chose is how a table gets
dropped, which is why this is the first thing any book about databases says.

### The fix, which is one character and not an argument for anything

`?` where the value goes, and the value beside the query:


In [3]:
print("with a placeholder:", connection.execute(
    "SELECT id, name FROM author WHERE name = ?", ("Ines O'Brien",)).fetchall())


with a placeholder: [(2, "Ines O'Brien")]


That is the whole fix, it is in the standard library, and it needs no mapper at all. The
**sqlite3, Deep Dive** guide owns it, along with what the driver does with the values it is handed.

So the case for peewee has to be made against this line rather than against the broken one. What
follows is what that line still leaves you doing by hand.

### What is left: a row that is an object

The row that came back is a tuple, and everything read from it is read by position:


In [4]:
row = connection.execute("SELECT id, name FROM author WHERE name = ?", ("Ines O'Brien",)).fetchone()
print("a tuple:", row, "| the name is", row[1])

db = SqliteDatabase(":memory:")


class Author(Model):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()

    class Meta:
        database = db


class Book(Model):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()

    class Meta:
        database = db


db.create_tables([Author, Book])
for name, year in AUTHORS:
    Author.create(name=name, first_book=year)

found = Author.get(Author.name == "Ines O'Brien")
print("an object:", found.name, "| first book", found.first_book, "|", type(found).__name__)


a tuple: (2, "Ines O'Brien") | the name is Ines O'Brien
an object: Ines O'Brien | first book 1998 | Author


`row[1]` says nothing about what it holds, and a column added at the front of that `SELECT` moves
every number in the code that reads it. `found.name` says what it is, and a new column changes
nothing that does not use it.

### A schema written once

The class is the table. peewee writes the `CREATE TABLE` from the fields, and the same declaration
is what the queries are built from and what the **Migrations** notebook compares a database against:


In [5]:
print(Author._schema._create_table().query()[0])
print(Book._schema._create_table().query()[0])


CREATE TABLE IF NOT EXISTS "author" ("id" INTEGER NOT NULL PRIMARY KEY, "name" VARCHAR(60) NOT NULL, "first_book" INTEGER NOT NULL)
CREATE TABLE IF NOT EXISTS "book" ("id" INTEGER NOT NULL PRIMARY KEY, "title" VARCHAR(80) NOT NULL, "author_id" INTEGER NOT NULL, "year" INTEGER NOT NULL, "pages" INTEGER NOT NULL, FOREIGN KEY ("author_id") REFERENCES "author" ("id"))


Two classes, two tables, and nothing written twice. The `unique=True` on the author's name is in the
table, the `index=True` on a book's year is a separate `CREATE INDEX`, and the foreign key is a
column and a constraint, which the **Models and Fields** notebook takes apart.

### A related row reached by attribute

The catalog, loaded, and a book asked which author wrote it:


In [6]:
written = {author.name: author for author in Author.select()}
for title, author, year, pages in BOOKS:
    Book.create(title=title, author=written[author], year=year, pages=pages)

book = Book.get(Book.title == "Winter Harbour")
print("the book  :", book.title, book.year)
print("its author:", book.author.name)
print("and back  :", sorted(other.title for other in book.author.books))


the book  : Winter Harbour 2011
its author: Ines O'Brien
and back  : ['A Careful Fire', 'The Long Field', 'Winter Harbour']


`book.author` is the row in the other table, fetched when it is read, and `author.books` is the list
of books pointing back, which is what `backref="books"` on the field created. Neither is free: each
of those is a query, and the **Relationships** and **prefetch and Load** notebooks are about
counting them and about the two ways to stop reading them one at a time.

### Where SQLAlchemy is the better answer

peewee is small on purpose, and the things it leaves out are the reasons to reach for something
else:

| What you need | peewee | SQLAlchemy |
|---|---|---|
| a migration written from the models | `pwmigrate diff` and `generate`, new in 4.5 | Alembic's autogenerate, against a live database |
| branching migration history, several heads merged | one line of revisions | branches, merges and `heads` |
| a query built from pieces, or SQL that is not a `SELECT` of one table | the query builder, and `raw` past its edge | Core, which is a full expression language |
| four databases from one codebase | two backends, SQLite and PostgreSQL, in this guide | a dialect for each, compiled without connecting |
| asynchronous work | out of scope here | the async engine, and the drivers that go with it |

The **SQLAlchemy, Deep Dive** guide is that library, and the **asyncpg and psycopg3, Deep Dive**
guide is where asynchronous database work belongs. What peewee gives in return is on the screen
above: one module, no dependencies, a class that is a table, and a query you can print.

### A catalog in fifteen lines, finished

The whole of this notebook as a program: a schema, a load, a query with a value in it, and an answer
read by name:


In [7]:
def longest_books(minimum_pages):
    """Every book over a length, newest first, with the author's name beside it."""
    found = (Book.select()
             .where(Book.pages > minimum_pages)
             .order_by(Book.year.desc()))
    return [(book.title, book.year, book.author.name) for book in found]


print("the query:", sql(Book.select().where(Book.pages > 300).order_by(Book.year.desc())))
for title, year, author in longest_books(300):
    print(f"  {title:<20} {year}  {author}")


the query: SELECT "t1"."id", "t1"."title", "t1"."author_id", "t1"."year", "t1"."pages" FROM "book" AS "t1" WHERE ("t1"."pages" > ?) ORDER BY "t1"."year" DESC  [300]
  The Quiet Engine     2021  Ursula Vance
  Harmattan            2019  Kofi Mensah
  The Salt Road        2014  Ursula Vance
  Stone and Tide       2009  Marco Pietra
  The Long Field       2004  Ines O'Brien
  A Careful Fire       1998  Ines O'Brien


Six books, newest first, each with the author it belongs to. The query printed above it is what
peewee sent: the 300 is a value beside the SQL rather than text inside it, the ordering is a clause
rather than a sort in Python, and the author's name came from an attribute.

### Where each part came from

| In `longest_books` | What it relies on | The section that showed it |
|---|---|---|
| `Book.select().where(...)` | a query built from the model, with values kept out of the text | A query built by hand |
| `book.title` and `book.year` | a row that is an object | What is left: a row that is an object |
| `book.author.name` | a foreign key read as an attribute | A related row reached by attribute |
| `sql(...)` | a query printable before it runs | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/01-why-peewee-solutions.ipynb).

**1.** Print the SQL of a query for every book of 1998 or later, and then the titles it returns.


In [8]:
# your code here


**2.** Print every author's name beside the number of books the catalog has for them, using the
`books` attribute.


In [9]:
# your code here


**3.** Build the same query as task 1 by hand with `sqlite3`, using a placeholder, and show it
returns the same number of rows.


In [10]:
# your code here


**4.** Print the `CREATE TABLE` peewee writes for a new model of your own with a name, a count and a
date, and say in a comment which line made the index.


In [11]:
# your code here


**5.** Find the book with the most pages without sorting in Python, and print its title and author.


In [12]:
# your code here


**6.** Write `books_by(name)`, returning the titles an author wrote, in year order, and show what it
gives for a name that is not in the catalog.


In [13]:
# your code here


## Common errors

### AttributeError: type object 'Author' has no attribute 'objects'


In [14]:
Author.objects.filter(name="Ursula Vance")


AttributeError: type object 'Author' has no attribute 'objects'

That is Django's spelling. peewee has no manager object in the middle: the query starts on the model
itself, `Author.select()`, and the condition goes in `where` rather than in keyword arguments.

The one that catches people next is the same habit in the condition,
`Author.select().where(name="Ursula Vance")`, which peewee reads as a keyword argument it has no use
for. A condition is an expression built from the field:


In [15]:
print("as an expression:", sql(Author.select().where(Author.name == "Ursula Vance")))
print("the author      :", Author.get(Author.name == "Ursula Vance").first_book)


as an expression: SELECT "t1"."id", "t1"."name", "t1"."first_book" FROM "author" AS "t1" WHERE ("t1"."name" = ?)  ['Ursula Vance']
the author      : 2014


### TypeError: 'Author' object is not subscriptable


In [16]:
first = Author.get_by_id(1)
print(first[0])


TypeError: 'Author' object is not subscriptable

That is the standard library's habit: `cursor.fetchone()` gives a tuple and a tuple is indexed. What
peewee returns is an instance of the model, so its columns are attributes, and the id is `first.id`
rather than `first[0]`.

Where a tuple really is wanted, ask for one, which the **Selecting Rows** notebook covers:


In [17]:
print("as an object:", first.id, first.name)
print("as a tuple  :", Author.select().tuples().first())
print("as a dict   :", Author.select().dicts().first())


as an object: 1 Ursula Vance
as a tuple  : (1, 'Ursula Vance', 2014)
as a dict   : {'id': 1, 'name': 'Ursula Vance', 'first_book': 2014}


### peewee.OperationalError: near "Brien": syntax error


In [18]:
db.execute_sql("SELECT id, name FROM author WHERE name = '%s'" % "Ines O'Brien")


OperationalError: near "Brien": syntax error

The same broken query as the first worked example, sent through peewee this time, and it fails in
exactly the same way. That is worth seeing: nothing about having a mapper installed protects a
string you built yourself. What protects you is the query builder, and `execute_sql` is the door out
of it.

The door is there for a reason, and the way through it keeps the values out of the text:


In [19]:
cursor = db.execute_sql("SELECT id, name FROM author WHERE name = ?", ("Ines O'Brien",))
print("through the door:", cursor.fetchall())
print("as models       :", [author.name for author in
                            Author.raw("SELECT * FROM author WHERE name = ?", "Ines O'Brien")])


through the door: [(3, "Ines O'Brien")]
as models       : ["Ines O'Brien"]


## Recap

- A value built into a query's text breaks on an apostrophe and is how a database gets emptied; a
  placeholder fixes it, and that fix is the standard library's rather than any mapper's.
- What a mapper adds is what the fix leaves alone: a row that is an object, a schema written once,
  and a related row reached by attribute.
- A model class is the table, and peewee writes the `CREATE TABLE` from its fields.
- Every query can be printed as the SQL it will send, with `.sql()`, which is what makes this
  guide's claims checkable.
- SQLAlchemy is the better answer for branching migrations, for queries past a single table's
  builder, and for asynchronous work.


## What is next

The **Models and Fields** notebook is the model class itself: `Meta.database` and the base class
every peewee program starts with, what each field becomes as a column, unique constraints and
indexes over two columns, and the two failures of the first hour, a model bound to no database and a
model whose table was never made.


---

[Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
